# 基于 MGWR 的美国县域 COPD 患病率空间异质性分析

## 1. 研究问题

本实验聚焦的问题是：

**在控制吸烟率与肥胖率之后，2019 年 PM2.5 年均浓度对美国县域 COPD 患病率的影响是否存在显著空间异质性？如果存在，这种影响在不同地区的方向和强度是否一致？**

这一问题适合用 MGWR 来回答，因为 MGWR 不仅能估计变量的整体关系，还能揭示不同变量在不同空间尺度上的作用差异。


## 2. 数据来源与变量

### 2.1 数据来源

1. **空气质量数据**：Hugging Face 数据集 `meganariley/daily-aqi`
   - 提取 2019 年 `pm25` 监测记录
   - 采用“站点日均值 -> 县域日均值 -> 县域年均值”的方式聚合

2. **健康数据**：CDC PLACES County Data, 2021 release
   - 多数指标基于 2019 年 BRFSS 建模
   - 与 2019 年 PM2.5 数据具有较好的时间匹配性

### 2.2 变量设定

- 因变量：`COPD_AdjPrev`
- 核心解释变量：`pm25_2019_annual_mean`
- 控制变量：`CSMOKING_AdjPrev`
- 控制变量：`OBESITY_AdjPrev`

最终进入 MGWR 的样本为 **401 个县**。


## 3. 模型方法与参数设置

本实验先构建 OLS 作为基准模型，再使用 MGWR 估计局部关系。

MGWR 参数设置如下：

- Coordinates type: `Spherical`
- GWR mode: `MGWR`
- Kernel: `Adaptive bisquare`
- Bandwidth searching: `Golden Section`
- Model type: `Gaussian`
- Optimization criterion: `AICc`

该设置允许不同变量具有不同带宽，是识别多尺度空间异质性的关键。


## 4. 全局回归与整体 MGWR 结果

根据本次作图摘要结果：

| 指标 | 数值 |
|---|---:|
| OLS R² | 0.8005 |
| OLS Adjusted R² | 0.7990 |
| PM2.5 OLS coefficient | 0.0672 |
| Smoking OLS coefficient | 0.8160 |
| Obesity OLS coefficient | 0.1012 |

结合 MGWR summary 文件可知：

| 指标 | OLS | MGWR |
|---|---:|---:|
| R² | 0.800 | 0.954 |
| Adjusted R² | 0.799 | 0.946 |
| AICc | 501.778 | 48.265 |
| RSS | 80.005 | 18.643 |

这表明 MGWR 相比 OLS 取得了非常明显的拟合改善，说明县域 COPD 患病率的影响机制具有显著空间非平稳性。


## 5. 图 1：OLS 基准诊断

![OLS diagnostics](./output_01_ols_diagnostics.png)

这张图主要用于说明 OLS 并不是一个完全失效的模型。换句话说，本实验使用 MGWR 不是因为 OLS 完全不能解释数据，而是因为在一个已经有较高解释力的全局模型之上，仍然存在值得进一步识别的局部差异。


## 6. 图 2：局部拟合优度（Local R²）

![Local R2](./output_02_local_r2.png)

MGWR 的局部拟合优度统计结果为：

- 平均值：**0.9123**
- 中位数：**0.9221**

说明 MGWR 在绝大多数地区都能较好解释 COPD 的空间分布。不过，这里的高 `localR2` 更适合被理解为“模型整体拟合较好”，而不是“每个解释变量在每个地方都同样重要”。


## 7. 图 3：局部系数空间分布

![Local coefficients](./output_03_local_coefficients.png)

这组图并排展示了 PM2.5、吸烟率和肥胖率的局部系数空间分布，是理解 MGWR 结果的核心图件之一。

可以观察到：

- **吸烟率系数**在全国范围内稳定为正，且数值整体较高；
- **肥胖率系数**也稳定为正，但空间波动幅度明显更小；
- **PM2.5 系数**则同时出现正值与负值，空间差异最明显。

这说明三类变量并不在同一空间尺度上发挥作用。


## 8. 图 4：PM2.5 显著性空间分布

![PM2.5 significance](./output_04_pm25_significance.png)

这张图将 PM2.5 的局部效应分成三类：

- 显著正向
- 显著负向
- 不显著

根据结果统计：

- PM2.5 显著县数：**112 / 401**
- 显著正向：**48**
- 显著负向：**64**

因此，PM2.5 并不是一个在全国范围内普遍同向显著的变量。它的空间作用具有很强的局部性，而且方向并不一致。

这一点非常重要：  
**PM2.5 的局部负系数不能直接理解为保护效应，更合理的解释是，在控制吸烟率和肥胖率后，PM2.5 的条件关联在不同地区被地方背景因素重新塑造。**


## 9. 图 5：MGWR 局部系数与 OLS 全局系数对比

![GWR vs OLS](./output_05_gwr_vs_ols.png)

这一组箱线图把 MGWR 的局部系数分布与 OLS 的全局系数放到同一张图里对照，能非常直观地看出：

- 吸烟率的局部系数虽然有波动，但始终围绕较高的正值分布；
- 肥胖率的局部系数分布非常集中；
- PM2.5 的局部系数分布跨度最大，并且穿越正负两侧。

这说明 OLS 给出的只是“全国平均效应”，而 MGWR 揭示了这一平均效应背后复杂的地方差异。


## 10. 图 6：MGWR 相对 OLS 的改进

![R2 improvement](./output_06_r2_improvement.png)

该图展示了 `local R² - global OLS R²` 的空间分布，用于说明 MGWR 相比 OLS 的提升主要出现在哪些地区。

从结果看，绝大多数地区都表现为 MGWR 优于 OLS，说明即便 OLS 的整体 R² 已经较高，空间非平稳性依然是不可忽略的。


## 11. 图 7：局部系数频率分布

![Coefficient distributions](./output_07_coef_distributions.png)

这一组直方图进一步说明了三个变量的分布特征：

- **吸烟率**：局部系数主要集中在较高正值区间，是最稳健的核心变量；
- **肥胖率**：分布最集中，说明它更像一个全局性变量；
- **PM2.5**：分布最分散，且跨越正负两侧，是空间异质性最强的变量。


## 12. 多尺度解释：全局变量与局部变量

根据 MGWR summary 文件，本次结果的变量带宽如下：

| 变量 | 带宽 |
|---|---:|
| Intercept | 43 |
| PM2.5 | 50 |
| Smoking | 46 |
| Obesity | 387 |

这一结果非常值得强调：

- **肥胖率带宽高达 387**，几乎接近总样本量 401，说明它更像一个宏观、均质的全局变量；
- **PM2.5 和吸烟率带宽都在 40–50 左右**，说明它们的影响更偏局部，且不同地区之间变化更剧烈。

从这一点看，MGWR 的最大科学价值就在于：  
**同一个模型中的不同变量并不共享同一个空间尺度。**


## 13. 讨论与局限

本实验虽然结果较完整，但仍存在以下局限：

1. 样本不是美国全部县，而是空气质量监测和健康数据都可用的样本县；
2. PM2.5 只有约 27.9% 的县达到显著，说明它的重要性并不意味着其作用在全国都稳定；
3. PM2.5 的负系数更可能反映局部条件关联变化、遗漏变量影响或区域背景差异，而不应被直接解释为保护效应；
4. MGWR 的高拟合优度需要谨慎解读，报告中不能只强调 `R²` 提升，更应强调空间尺度差异和局部显著性分布。


## 14. 结论

基于 2019 年 PM2.5 数据与 CDC PLACES 县级健康数据，本文对美国县域 COPD 患病率进行了 MGWR 分析。主要结论如下：

1. MGWR 显著优于 OLS，说明 COPD 的空间形成机制具有明显异质性；
2. 吸烟率是最稳定、最强的正向影响因素；
3. 肥胖率也呈稳定正向作用，但更接近全局尺度变量；
4. PM2.5 是最有空间异质性的变量，在部分地区显著为正，在部分地区显著为负，且并非全国普遍显著。

因此，本次实验最重要的发现并不是“PM2.5 在全国一致性提高 COPD 风险”，而是：

> **PM2.5 对 COPD 的影响具有明显的空间非平稳性，其方向和强度依赖于地方背景条件；相比之下，吸烟率和肥胖率的影响更加稳定。**
